In [15]:
import os
import re
from io import BytesIO
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup


# --------------------------------
# SMALL UTILITIES
# --------------------------------
def format_filename(text):
    text = str(text).strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_")


def get_excel_basename(source):
    parsed = urlparse(str(source))
    if parsed.scheme in ("http", "https"):
        return os.path.basename(parsed.path)
    return os.path.basename(str(source))


def clean_path(path):
    return path.strip().strip('"').strip("'")


def is_blank(x):
    return pd.isna(x) or str(x).strip() == "" or str(x).strip().lower() in {"nan", "none"}


def is_number_like(x):
    if pd.isna(x):
        return False

    if isinstance(x, (int, float)) and not pd.isna(x):
        return True

    s = str(x).strip()
    if not s:
        return False

    s = s.replace(",", "").replace("%", "").replace("€", "").replace("£", "")

    try:
        float(s)
        return True
    except Exception:
        return False


def clean_numeric_series(series):
    s = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("€", "", regex=False)
        .str.replace("£", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})
    )
    return pd.to_numeric(s, errors="coerce")


def make_unique(cols):
    seen = {}
    out = []
    for c in cols:
        c = str(c).strip()
        if c in seen:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            out.append(c)
    return out


def extract_year(text):
    m = re.search(r"\b((?:19|20)\d{2})\b", str(text))
    return m.group(1) if m else None


def extract_year_from_header(text):
    m = re.search(r"\b((?:19|20)\d{2})\b", str(text))
    return m.group(1) if m else None


def trim_empty_edges(df):
    return df.dropna(how="all").dropna(axis=1, how="all")


def looks_like_footer(value):
    if pd.isna(value):
        return False

    s = str(value).strip().lower()
    footer_terms = [
        "coverage",
        "source",
        "note",
        "notes",
        "total coverage",
        "data source",
        "prepared by",
    ]
    return any(term in s for term in footer_terms)


def looks_like_footer_row(row_values):
    footer_terms = [
        "coverage",
        "source",
        "note",
        "notes",
        "total coverage",
        "data source",
        "prepared by",
        "methodology",
        "definitions",
        "disclaimer",
    ]
    joined = " ".join([str(v).strip().lower() for v in row_values if not is_blank(v)])
    return any(term in joined for term in footer_terms)


def looks_like_date_or_period(value):
    if pd.isna(value):
        return False

    s = str(value).strip()

    if re.fullmatch(r"\d{4}", s):
        return True

    if re.fullmatch(r"Q[1-4]\s*\d{4}", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"\d{4}\s*Q[1-4]", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"[A-Za-z]{3,9}\s+\d{4}", s):
        return True

    try:
        pd.to_datetime(s, errors="raise")
        return True
    except Exception:
        return False


def is_year_col(col):
    if pd.isna(col):
        return False

    if isinstance(col, (int, float)):
        try:
            y = int(col)
            return 1900 <= y <= 2100
        except Exception:
            return False

    s = str(col).strip()
    if re.fullmatch(r"\d{4}", s):
        return True

    return bool(re.search(r"\b(19|20)\d{2}\b", s))


def is_year_header(x):
    if pd.isna(x):
        return False

    if isinstance(x, (int, float)):
        try:
            y = int(x)
            return 1900 <= y <= 2100
        except Exception:
            return False

    s = str(x).strip()
    if re.fullmatch(r"(19|20)\d{2}", s):
        return True

    return bool(re.search(r"\b(19|20)\d{2}\b", s))


# --------------------------------
# FINAL OUTPUT CLEANING
# --------------------------------
def standardise_final_schema(df):
    """
    Standardise a few common names, but do NOT remove real analytical columns.
    """
    df = df.copy()
    rename_map = {}

    existing_lower = {str(c).strip().lower() for c in df.columns}

    for col in df.columns:
        col_str = str(col).strip()
        col_low = col_str.lower()

        if col_low == "variable" and "measure" not in existing_lower:
            rename_map[col] = "measure"
        elif col_low == "values":
            rename_map[col] = "value"
        elif col_low == "amount" and "value" not in existing_lower:
            rename_map[col] = "value"
        elif col_low == "cover type":
            rename_map[col] = "CoverType"
        elif col_low == "accident quarter":
            rename_map[col] = "AccidentQuarter"
        elif col_low == "origin year":
            rename_map[col] = "OriginYear"
        elif col_low == "underwriting year":
            rename_map[col] = "UnderwritingYear"
        elif col_low == "claim type":
            rename_map[col] = "ClaimType"
        elif col_low == "year":
            rename_map[col] = "year"
        elif col_low == "measure":
            rename_map[col] = "measure"
        elif col_low == "value":
            rename_map[col] = "value"

    if rename_map:
        df = df.rename(columns=rename_map)

    return df


def ensure_required_time_and_value(df):
    """
    Require:
    - a value column
    - at least one time-related column
    Do NOT remove useful dimensional columns like CoverType.
    """
    df = df.copy()
    df = standardise_final_schema(df)

    cols_lower = {str(c).strip().lower(): c for c in df.columns}

    if "year" in cols_lower and cols_lower["year"] != "year":
        df = df.rename(columns={cols_lower["year"]: "year"})
        cols_lower = {str(c).strip().lower(): c for c in df.columns}

    if "date" in cols_lower and cols_lower["date"] != "date":
        df = df.rename(columns={cols_lower["date"]: "date"})
        cols_lower = {str(c).strip().lower(): c for c in df.columns}

    if "value" in cols_lower and cols_lower["value"] != "value":
        df = df.rename(columns={cols_lower["value"]: "value"})
        cols_lower = {str(c).strip().lower(): c for c in df.columns}

    # If date is basically a year field, also create year
    if "date" in df.columns and "year" not in df.columns:
        possible_years = df["date"].astype(str).str.extract(r"((?:19|20)\d{2})", expand=False)
        non_null_ratio = possible_years.notna().mean() if len(possible_years) else 0
        if non_null_ratio >= 0.8:
            df["year"] = possible_years

    if "value" not in df.columns:
        return pd.DataFrame()

    time_like_cols = {
        "year",
        "date",
        "quarter",
        "accidentquarter",
        "originyear",
        "underwritingyear",
    }

    present_time_cols = [c for c in df.columns if str(c).strip().lower() in time_like_cols]

    if not present_time_cols:
        return pd.DataFrame()

    return df


def drop_metadata_columns(df):
    """
    Drop only metadata/debug columns.
    Keep all analytical columns such as:
    year, AccidentQuarter, CoverType, ClaimType, section, dimensions, etc.
    """
    df = df.copy()
    df = ensure_required_time_and_value(df)

    if df.empty:
        return df

    metadata_cols = {
        "source_file",
        "sheet_name",
        "table_index",
        "table_title",
        "processing_method",
        "output_file",
        "method_used",
        "status",
    }

    keep_cols = [c for c in df.columns if str(c).strip().lower() not in metadata_cols]

    if not keep_cols:
        return pd.DataFrame()

    return df[keep_cols]


# --------------------------------
# SCRIPT 1 LOGIC
# --------------------------------
def detect_header_rows(df, max_scan_rows=10, max_header_rows=4):
    header_rows = []

    for i in range(min(max_scan_rows, len(df))):
        row = df.iloc[i]
        values = [x for x in row if not is_blank(x)]

        if len(values) == 0:
            continue
        if len(values) == 1:
            continue  # likely title row

        text_cells = sum(not is_number_like(x) for x in values)
        text_ratio = text_cells / len(values)

        if text_ratio >= 0.6:
            header_rows.append(i)
            if len(header_rows) >= max_header_rows:
                break
        else:
            if header_rows:
                break

    return header_rows if header_rows else [0]


def clean_combined_headers(header_block):
    header_block = header_block.ffill(axis=1)

    new_columns = []
    for col in header_block.columns:
        parts = []
        for val in header_block[col]:
            if not is_blank(val):
                part = str(val).strip()
                if part not in parts:
                    parts.append(part)

        if parts:
            new_columns.append(" | ".join(parts))
        else:
            new_columns.append(f"column_{col}")

    return make_unique(new_columns)


def find_year_row_index(df, max_scan=25):
    scan_rows = min(max_scan, len(df))
    best_row_idx = None
    best_year_count = 0

    for i in range(scan_rows):
        row_vals = [str(x).strip() for x in df.iloc[i].tolist()]
        year_count = sum(bool(re.fullmatch(r"(19|20)\d{2}", v)) for v in row_vals)

        if year_count > best_year_count:
            best_year_count = year_count
            best_row_idx = i

    return best_row_idx, best_year_count


def wide_years_to_long(df, year_cols):
    non_year_cols = [c for c in df.columns if c not in year_cols]
    if not non_year_cols:
        return pd.DataFrame()

    rename_map = {non_year_cols[0]: "measure"}
    id_vars = ["measure"]

    if len(non_year_cols) >= 2:
        rename_map[non_year_cols[1]] = "measure_id"
        id_vars.append("measure_id")

    for k in range(2, len(non_year_cols)):
        dim_name = f"dimension_{k+1}"
        rename_map[non_year_cols[k]] = dim_name
        id_vars.append(dim_name)

    df = df.rename(columns=rename_map)

    df = df[~df["measure"].apply(looks_like_footer)].reset_index(drop=True)
    if df.empty:
        return pd.DataFrame()

    for yc in year_cols:
        df[yc] = clean_numeric_series(df[yc])

    long_df = df.melt(
        id_vars=id_vars,
        value_vars=year_cols,
        var_name="year",
        value_name="value"
    )
    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)
    long_df["year"] = long_df["year"].apply(extract_year_from_header)
    long_df = long_df.dropna(subset=["year"]).reset_index(drop=True)

    return long_df


def tidy_dataframe_to_long(df):
    """
    Script 1 tidy logic.
    """
    df = df.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
    if df.empty:
        return df

    yr_idx, yr_count = find_year_row_index(df, max_scan=25)
    if yr_idx is not None and yr_count >= 3:
        promoted = df.iloc[yr_idx].tolist()
        new_cols = []
        for j, v in enumerate(promoted):
            v = str(v).strip()
            if v and v.lower() not in {"nan", "none"}:
                new_cols.append(v)
            else:
                new_cols.append(f"column_{j}")
        new_cols = make_unique(new_cols)

        df2 = df.iloc[yr_idx + 1:].reset_index(drop=True)
        if not df2.empty:
            df2.columns = [str(c).strip() for c in new_cols[:df2.shape[1]]]
            year_cols = [c for c in df2.columns if is_year_col(c)]
            if len(year_cols) >= 3:
                return wide_years_to_long(df2, year_cols)

    header_rows = detect_header_rows(df)
    header_block = df.iloc[header_rows].copy()
    new_columns = clean_combined_headers(header_block)

    data_start = max(header_rows) + 1
    df = df.iloc[data_start:].reset_index(drop=True)
    if df.empty:
        return df

    df.columns = [str(c).strip() for c in new_columns[:df.shape[1]]]
    df = df.dropna(how="all").reset_index(drop=True)
    if df.empty:
        return df

    # Already-long detection
    lower_to_orig = {c.lower(): c for c in df.columns}
    value_col = None
    for key in ["value", "values", "amount"]:
        if key in lower_to_orig:
            value_col = lower_to_orig[key]
            break

    if value_col is not None:
        df[value_col] = clean_numeric_series(df[value_col])
        df = df.dropna(subset=[value_col]).reset_index(drop=True)

        if value_col != "value":
            df = df.rename(columns={value_col: "value"})

        return df

    # Years as columns
    year_cols = [c for c in df.columns if is_year_col(c)]
    if len(year_cols) >= 3:
        return wide_years_to_long(df, year_cols)

    # Dates / periods in first column
    first_col = df.columns[0]

    df = df[~df[first_col].apply(looks_like_footer)].reset_index(drop=True)
    if df.empty:
        return df

    df = df.rename(columns={first_col: "date"})
    df = df[df["date"].apply(looks_like_date_or_period)].reset_index(drop=True)
    if df.empty:
        return df

    other_cols = [c for c in df.columns if c != "date"]

    melt_cols = []
    for col in other_cols:
        sample = df[col].dropna().head(30)
        if len(sample) == 0:
            continue

        numeric_ratio = sum(is_number_like(x) for x in sample) / len(sample)

        if numeric_ratio >= 0.7:
            df[col] = clean_numeric_series(df[col])
            if df[col].notna().sum() > 0:
                melt_cols.append(col)
        else:
            df[col] = df[col].astype(str).str.strip().replace({"": pd.NA})

    if not melt_cols:
        return pd.DataFrame()

    long_df = df.melt(
        id_vars=["date"],
        value_vars=melt_cols,
        var_name="variable",
        value_name="value"
    )
    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)
    return long_df


# --------------------------------
# SCRIPT 2 LOGIC
# --------------------------------
def drop_title_rows_at_top(df, max_drop=12):
    df = df.copy().reset_index(drop=True)
    drops = 0

    while len(df) > 0 and drops < max_drop:
        row = df.iloc[0].tolist()
        vals = [v for v in row if not is_blank(v)]

        if len(vals) <= 2 and all(not is_number_like(v) for v in vals):
            df = df.iloc[1:].reset_index(drop=True)
            drops += 1
        else:
            break

    return df


def split_by_blank_rows(df, blank_run=1, min_rows=4):
    df = df.copy()
    blank = df.apply(lambda r: all(is_blank(v) for v in r.values), axis=1).tolist()

    blocks = []
    start = None
    run = 0

    for i, is_b in enumerate(blank):
        if not is_b:
            if start is None:
                start = i
            run = 0
        else:
            if start is not None:
                run += 1
                if run >= blank_run:
                    end = i - run + 1
                    block = df.iloc[start:end].copy()
                    block = trim_empty_edges(block)
                    if len(block) >= min_rows and block.shape[1] >= 2:
                        blocks.append((block, start, end))
                    start = None
                    run = 0

    if start is not None:
        end = len(df)
        block = df.iloc[start:end].copy()
        block = trim_empty_edges(block)
        if len(block) >= min_rows and block.shape[1] >= 2:
            blocks.append((block, start, end))

    return blocks


def split_by_blank_cols(df, blank_run=1, min_cols=2):
    df = df.copy()
    blank_cols = []

    for c in range(df.shape[1]):
        col_vals = df.iloc[:, c].values
        blank_cols.append(all(is_blank(v) for v in col_vals))

    blocks = []
    start = None
    run = 0

    for j, is_b in enumerate(blank_cols):
        if not is_b:
            if start is None:
                start = j
            run = 0
        else:
            if start is not None:
                run += 1
                if run >= blank_run:
                    end = j - run + 1
                    block = df.iloc[:, start:end].copy()
                    block = trim_empty_edges(block)
                    if block.shape[1] >= min_cols and len(block) >= 2:
                        blocks.append((block, start, end))
                    start = None
                    run = 0

    if start is not None:
        end = df.shape[1]
        block = df.iloc[:, start:end].copy()
        block = trim_empty_edges(block)
        if block.shape[1] >= min_cols and len(block) >= 2:
            blocks.append((block, start, end))

    return blocks


def is_title_row(row_vals):
    nonblank = [v for v in row_vals if not is_blank(v)]
    if len(nonblank) == 0:
        return False

    if len(nonblank) <= 2 and all(not is_number_like(v) for v in nonblank):
        text = " ".join([str(v).lower() for v in nonblank])

        title_keywords = [
            "accompanies",
            "ncid",
            "private motor report",
            "figure",
            "table ",
            "breakdown of",
            "income and expenditure",
            "settled claimant",
            "settled claim",
        ]
        return any(k in text for k in title_keywords)

    return False


def split_on_title_rows(df, min_rows=4):
    df = df.copy().reset_index(drop=True)
    blocks = []
    start = 0
    current_title = None

    title_at = {}
    for i in range(len(df)):
        if is_title_row(df.iloc[i].values):
            nonblank = [v for v in df.iloc[i].values if not is_blank(v)]
            title_at[i] = str(nonblank[0]).strip() if nonblank else None

    for i in range(len(df)):
        if i in title_at and i != start:
            block = trim_empty_edges(df.iloc[start:i].copy())
            if len(block) >= min_rows and block.shape[1] > 2:
                blocks.append((block, start, i, current_title))
            start = i
            current_title = title_at[i]

        if i in title_at and start == i:
            current_title = title_at[i]

    block = trim_empty_edges(df.iloc[start:].copy())
    if len(block) >= min_rows and block.shape[1] > 2:
        blocks.append((block, start, len(df), current_title))

    if not blocks:
        block = trim_empty_edges(df.copy())
        if len(block) >= min_rows and block.shape[1] > 2:
            blocks = [(block, 0, len(df), None)]

    return blocks


def pick_header_row(df, scan=18):
    best_idx = 0
    best_score = -1

    for i in range(min(scan, len(df))):
        row = df.iloc[i].values
        vals = [v for v in row if not is_blank(v)]
        if len(vals) < 3:
            continue

        text_ct = sum(not is_number_like(v) for v in vals)
        text_ratio = text_ct / max(1, len(vals))
        year_ct = sum(bool(extract_year(v)) for v in vals)

        score = (len(vals) * 2) + (text_ratio * 5) + (min(year_ct, 10) * 1.5)
        if score > best_score:
            best_score = score
            best_idx = i

    return best_idx


def build_headers(df, header_row_idx, max_extra_rows=3):
    header_rows = [header_row_idx]

    for k in range(1, max_extra_rows + 1):
        idx = header_row_idx + k
        if idx >= len(df):
            break

        row = df.iloc[idx].values
        vals = [v for v in row if not is_blank(v)]
        if len(vals) < 3:
            break

        year_ct = sum(bool(extract_year(v)) for v in vals)
        text_ct = sum(not is_number_like(v) for v in vals)

        if year_ct >= 3 or (text_ct / len(vals)) >= 0.6:
            header_rows.append(idx)
        else:
            break

    header_block = df.iloc[header_rows].copy().ffill(axis=1)

    cols = []
    for c in range(header_block.shape[1]):
        parts = []
        for r in range(header_block.shape[0]):
            v = header_block.iat[r, c]
            if not is_blank(v):
                s = str(v).strip()
                if s not in parts:
                    parts.append(s)

        cols.append(" | ".join(parts) if parts else f"col_{c}")

    cols = make_unique(cols)
    data_start = max(header_rows) + 1
    return cols, data_start


def add_section_column_and_drop_section_rows(df):
    df = df.copy().reset_index(drop=True)

    current_section = None
    sections = []
    keep_rows = []

    for i in range(len(df)):
        row = df.iloc[i].tolist()
        nonblank = [v for v in row if not is_blank(v)]

        if 1 <= len(nonblank) <= 2 and all(not is_number_like(v) for v in nonblank):
            label = str(nonblank[0]).strip()
            if label and label.lower() not in {"years", "year"}:
                current_section = label
                continue

        sections.append(current_section)
        keep_rows.append(i)

    out = df.iloc[keep_rows].copy().reset_index(drop=True)
    out.insert(0, "section", sections)
    out["section"] = out["section"].replace({"Missing value": pd.NA, "": pd.NA})
    out["section"] = out["section"].ffill()
    return out


def remove_embedded_header_noise(df):
    df = df.copy().reset_index(drop=True)
    colnames = {str(c).strip().lower() for c in df.columns}

    header_terms = [
        "category",
        "measure",
        "histexp",
        "histexpmeasure",
        "calculation",
        "accompanies",
        "ncid",
        "private motor report",
        "figure",
        "table",
        "breakdown of",
    ]

    drop_idx = []
    for i in range(len(df)):
        row = df.iloc[i].tolist()
        nonblank = [v for v in row if not is_blank(v)]
        if len(nonblank) == 0:
            continue

        row_text = " ".join([str(v).strip().lower() for v in nonblank])
        numeric_ct = sum(is_number_like(v) for v in nonblank)
        hits = sum(1 for v in nonblank if str(v).strip().lower() in colnames)

        if (any(t in row_text for t in header_terms) and numeric_ct <= 1) or hits >= 2:
            drop_idx.append(i)

    return df.drop(index=drop_idx).reset_index(drop=True)


def standardise_measure_columns(df):
    df = df.copy()
    cols = list(df.columns)

    for c in cols:
        c_low = str(c).lower()
        if "measure" in c_low and c not in {"measure", "measure_id"}:
            df = df.rename(columns={c: "measure"})
            break

    for c in list(df.columns):
        c_low = str(c).lower()
        if ("id" in c_low) and c not in {"measure_id"} and c != "table_index":
            if c_low in {"histexpmeasureid", "measureid", "id", "calculation"}:
                df = df.rename(columns={c: "measure_id"})
                break

    return df


# --------------------------------
# SPLIT HORIZONTALLY MERGED TIDY OUTPUTS
# --------------------------------
def base_col_name(col):
    col = str(col).strip()
    m = re.match(r"^(.*?)(?:_(\d+))$", col)
    if m:
        return m.group(1).strip()
    return col


def is_block_anchor(col):
    b = base_col_name(col).strip().lower()
    return b in {
        "year",
        "date",
        "quarter",
        "accidentquarter",
        "accident quarter",
        "originyear",
        "origin year",
        "underwritingyear",
        "underwriting year",
        "measure",
        "value",
    }


def normalise_split_columns(df):
    out = df.copy()

    rename_map = {}
    for c in out.columns:
        new_c = base_col_name(c).strip()
        new_c_low = new_c.lower()

        if new_c_low == "value":
            new_c = "value"
        elif new_c_low == "measure":
            new_c = "measure"
        elif new_c_low == "cover type":
            new_c = "CoverType"
        elif new_c_low == "accident quarter":
            new_c = "AccidentQuarter"
        elif new_c_low == "claim type":
            new_c = "ClaimType"

        rename_map[c] = new_c

    out = out.rename(columns=rename_map)
    out.columns = make_unique(out.columns)

    if "value" in out.columns:
        out["value"] = clean_numeric_series(out["value"])
        out = out.dropna(subset=["value"]).reset_index(drop=True)

    return out


def split_merged_tidy_output(df):
    """
    If tidy_problem_table() accidentally merged two side-by-side tables into one dataframe,
    split them back into separate dataframes.
    """
    df = df.copy()
    cols = list(df.columns)

    if len(cols) <= 1:
        return [df]

    first_anchor_idx = None
    for i, c in enumerate(cols):
        if is_block_anchor(c):
            first_anchor_idx = i
            break

    if first_anchor_idx is None:
        return [df]

    shared_prefix = cols[:first_anchor_idx]
    rest = cols[first_anchor_idx:]

    seen_anchor_bases = set()
    block_starts = [0]

    for i, c in enumerate(rest):
        b = base_col_name(c).strip().lower()

        if is_block_anchor(c):
            if b in seen_anchor_bases:
                block_starts.append(i)
                seen_anchor_bases = {b}
            else:
                seen_anchor_bases.add(b)

    if len(block_starts) == 1:
        return [df]

    split_dfs = []
    for j, start in enumerate(block_starts):
        end = block_starts[j + 1] if j + 1 < len(block_starts) else len(rest)
        block_cols = rest[start:end]

        candidate_cols = shared_prefix + block_cols
        part = df[candidate_cols].copy()

        part = part.dropna(axis=1, how="all")
        part = normalise_split_columns(part)

        if not part.empty:
            split_dfs.append(part)

    return split_dfs if split_dfs else [df]


def tidy_problem_table(df):
    df = df.copy()

    df = trim_empty_edges(df).reset_index(drop=True)
    if df.empty or df.shape[1] < 2:
        return pd.DataFrame()

    df = drop_title_rows_at_top(df, max_drop=12)
    df = trim_empty_edges(df).reset_index(drop=True)
    if df.empty or len(df) < 3:
        return pd.DataFrame()

    cutoff = len(df)
    for i in range(len(df) - 1, max(-1, len(df) - 15), -1):
        if looks_like_footer_row(df.iloc[i].values):
            cutoff = i

    df = df.iloc[:cutoff].copy().reset_index(drop=True)
    df = trim_empty_edges(df).reset_index(drop=True)
    if df.empty:
        return pd.DataFrame()

    hdr = pick_header_row(df, scan=18)
    cols, data_start = build_headers(df, hdr, max_extra_rows=3)

    data = df.iloc[data_start:].copy().reset_index(drop=True)
    data = trim_empty_edges(data).reset_index(drop=True)
    if data.empty:
        return pd.DataFrame()

    data.columns = make_unique([str(c).strip() for c in cols[:data.shape[1]]])
    data = data.dropna(how="all").reset_index(drop=True)
    if data.empty:
        return pd.DataFrame()

    data = add_section_column_and_drop_section_rows(data)
    data = remove_embedded_header_noise(data)
    if data.empty:
        return pd.DataFrame()

    # Already long
    lower_to_orig = {c.lower(): c for c in data.columns}
    value_col = None
    for key in ["value", "values", "amount"]:
        if key in lower_to_orig:
            value_col = lower_to_orig[key]
            break

    if value_col is not None:
        out = data.copy()
        out[value_col] = clean_numeric_series(out[value_col])
        out = out.dropna(subset=[value_col]).reset_index(drop=True)
        if value_col != "value":
            out = out.rename(columns={value_col: "value"})
        out = standardise_measure_columns(out)
        return out

    # Years across columns
    year_cols = [c for c in data.columns if is_year_header(c)]
    if len(year_cols) >= 2:
        out = data.copy()
        for yc in year_cols:
            out[yc] = clean_numeric_series(out[yc])

        non_year_cols = [c for c in out.columns if c not in year_cols]
        long_df = out.melt(
            id_vars=non_year_cols,
            value_vars=year_cols,
            var_name="year",
            value_name="value"
        )
        long_df["year"] = long_df["year"].apply(extract_year)
        long_df = long_df.dropna(subset=["value", "year"]).reset_index(drop=True)
        long_df = standardise_measure_columns(long_df)
        return long_df

    # Generic melt
    numeric_cols = []
    for col in data.columns:
        if col == "section":
            continue

        sample = data[col].dropna().head(30)
        if len(sample) == 0:
            continue

        ratio = sum(is_number_like(x) for x in sample) / len(sample)
        if ratio >= 0.30:
            numeric_cols.append(col)

    if not numeric_cols:
        out = standardise_measure_columns(data)
        return out

    out = data.copy()
    for col in numeric_cols:
        out[col] = clean_numeric_series(out[col])

    id_vars = [c for c in out.columns if c not in numeric_cols]
    long_df = out.melt(
        id_vars=id_vars,
        value_vars=numeric_cols,
        var_name="variable",
        value_name="value"
    )
    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)
    long_df = standardise_measure_columns(long_df)
    return long_df


def process_with_script2(sheet_df, selected_source, sheet_name):
    """
    Runs Script 2 on one sheet and returns a LIST of tidy dataframes,
    one per detected table on that sheet.

    Also splits horizontally merged tidy outputs.
    """
    sheet_df = trim_empty_edges(sheet_df).reset_index(drop=True)

    if sheet_df.empty:
        return []

    big_blocks = split_on_title_rows(sheet_df, min_rows=4)

    table_idx = 0
    sheet_outputs = []

    for big_df, big_start, big_end, title_text in big_blocks:
        row_blocks = split_by_blank_rows(big_df, blank_run=1, min_rows=4)
        if not row_blocks:
            row_blocks = [(big_df, big_start, big_end)]

        for rb_df, rb_start, rb_end in row_blocks:
            col_blocks = split_by_blank_cols(rb_df, blank_run=1, min_cols=2)
            if not col_blocks:
                col_blocks = [(rb_df, 0, rb_df.shape[1])]

            for cb_df, cb_start, cb_end in col_blocks:
                try:
                    tidy_df = tidy_problem_table(cb_df)

                    if tidy_df is None or tidy_df.empty:
                        continue

                    split_tables = split_merged_tidy_output(tidy_df)

                    for split_df in split_tables:
                        if split_df is None or split_df.empty:
                            continue

                        table_idx += 1

                        # Keep metadata here for internal tracking;
                        # this gets removed before final CSV save.
                        split_df.insert(0, "source_file", get_excel_basename(selected_source))
                        split_df.insert(1, "sheet_name", sheet_name)
                        split_df.insert(2, "table_index", table_idx)
                        split_df.insert(3, "table_title", title_text)
                        split_df.insert(4, "processing_method", "script_2_fallback")

                        sheet_outputs.append(split_df)

                except Exception:
                    continue

    return sheet_outputs


# --------------------------------
# INPUT + SELECTION HELPERS
# --------------------------------
def choose_input_mode():
    print("Choose input method:")
    print("1. Scrape Excel files from a webpage")
    print("2. Use a single local Excel file")
    print("3. Use a folder of local Excel files")

    while True:
        choice = input("\nEnter 1, 2, or 3: ").strip()
        if choice in {"1", "2", "3"}:
            return choice
        print("Invalid choice.")


def choose_from_numbered_list(items, prompt):
    while True:
        try:
            choice = int(input(prompt).strip())
            if 1 <= choice <= len(items):
                return items[choice - 1]
            print("Please enter a valid number from the list.")
        except Exception:
            print("Please enter a valid number.")


def choose_sheets(sheet_names):
    print("\nSheets available:")
    print("-" * 40)
    for i, name in enumerate(sheet_names, start=1):
        print(f"{i}. {name}")
    print("-" * 40)
    print(f"Total sheets: {len(sheet_names)}")

    with open("sheet_list.txt", "w", encoding="utf-8") as f:
        for i, name in enumerate(sheet_names, start=1):
            f.write(f"{i}. {name}\n")
    print("Full sheet list saved to: sheet_list.txt")

    print("\nOptions:")
    print(" - Enter a single number (e.g. 2)")
    print(" - Enter multiple numbers separated by commas (e.g. 1,3,4)")
    print(" - Enter 'all' to process all sheets")

    while True:
        selection = input("\nSelect sheets: ").strip().lower()
        if selection == "all":
            return sheet_names

        try:
            indices = [int(x.strip()) for x in selection.split(",")]
            if all(1 <= i <= len(sheet_names) for i in indices):
                chosen = []
                for i in indices:
                    nm = sheet_names[i - 1]
                    if nm not in chosen:
                        chosen.append(nm)
                return chosen
            print("Invalid selection.")
        except Exception:
            print("Please enter valid numbers or 'all'.")


# --------------------------------
# MAIN
# --------------------------------
def main():
    headers = {"User-Agent": "Mozilla/5.0"}
    mode = choose_input_mode()

    selected_source = None
    df_dict = None

    # MODE 1: Web scrape
    if mode == "1":
        page_url = input("\nInsert webpage link: ").strip()

        response = requests.get(page_url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        excel_links = []
        for link in soup.find_all("a"):
            href = link.get("href")
            if href and (".xlsx" in href.lower() or ".xlsm" in href.lower()):
                excel_links.append(urljoin(page_url, href))

        excel_links = list(dict.fromkeys(excel_links))
        if not excel_links:
            print("No Excel files found on the webpage.")
            return

        print("\nExcel files found:\n")
        for i, link in enumerate(excel_links, start=1):
            print(f"{i}. {get_excel_basename(link)}")

        selected_source = choose_from_numbered_list(excel_links, "\nSelect file number: ")

        file_data = requests.get(selected_source, headers=headers)
        file_data.raise_for_status()

        df_dict = pd.read_excel(
            BytesIO(file_data.content),
            sheet_name=None,
            engine="openpyxl",
            header=None,
            dtype=object
        )

    # MODE 2: Local file
    elif mode == "2":
        selected_source = clean_path(input("\nEnter full file path: "))
        if not os.path.exists(selected_source):
            print("File not found. Check the path you pasted.")
            return

        df_dict = pd.read_excel(
            selected_source,
            sheet_name=None,
            engine="openpyxl",
            header=None,
            dtype=object
        )

    # MODE 3: Folder
    elif mode == "3":
        folder_path = clean_path(input("\nEnter folder path: "))
        if not os.path.isdir(folder_path):
            print("That folder path does not exist.")
            return

        files = [f for f in os.listdir(folder_path) if f.lower().endswith((".xlsx", ".xlsm"))]
        files.sort()

        if not files:
            print("No Excel files (.xlsx/.xlsm) found in that folder.")
            return

        print("\nExcel files available:\n")
        for i, f in enumerate(files, start=1):
            print(f"{i}. {f}")

        chosen_file = choose_from_numbered_list(files, "\nSelect file number: ")
        selected_source = os.path.join(folder_path, chosen_file)

        df_dict = pd.read_excel(
            selected_source,
            sheet_name=None,
            engine="openpyxl",
            header=None,
            dtype=object
        )

    else:
        print("Invalid mode.")
        return

    sheet_names = list(df_dict.keys())
    selected_sheets = choose_sheets(sheet_names)
    print("\nSelected sheets:", selected_sheets)

    os.makedirs("tidy_outputs", exist_ok=True)

    base_name = os.path.splitext(get_excel_basename(selected_source))[0]
    file_part = format_filename(base_name)

    log_rows = []

    for sheet_name in selected_sheets:
        print("\nProcessing sheet:", sheet_name)
        df = df_dict[sheet_name]

        sheet_part = format_filename(sheet_name)
        output_name = f"{file_part}__{sheet_part}.csv"
        output_path = os.path.join("tidy_outputs", output_name)

        try:
            # Try Script 1 first
            tidy_df = tidy_dataframe_to_long(df)

            if tidy_df is not None and not tidy_df.empty:
                if "processing_method" not in tidy_df.columns:
                    tidy_df.insert(0, "processing_method", "script_1")

                # Final clean before saving
                tidy_df = drop_metadata_columns(tidy_df)

                if tidy_df.empty:
                    print("  Script 1 produced output without required time/value columns -> trying Script 2...")
                    fallback_tables = process_with_script2(df, selected_source, sheet_name)
                else:
                    tidy_df.to_csv(output_path, index=False)

                    print(f"  Script 1 succeeded -> Saved: {output_name}")
                    print("  Output columns:", list(tidy_df.columns))

                    log_rows.append({
                        "source_file": get_excel_basename(selected_source),
                        "sheet_name": sheet_name,
                        "table_index": "",
                        "table_title": "",
                        "output_file": output_name,
                        "method_used": "script_1",
                        "status": "saved"
                    })
                    continue
            else:
                print("  Script 1 skipped -> trying Script 2...")
                fallback_tables = process_with_script2(df, selected_source, sheet_name)

            if fallback_tables:
                saved_any = False

                for i, table_df in enumerate(fallback_tables, start=1):
                    table_output_name = f"{file_part}__{sheet_part}__table_{i}.csv"
                    table_output_path = os.path.join("tidy_outputs", table_output_name)

                    table_index_val = ""
                    table_title_val = ""

                    if "table_index" in table_df.columns and not table_df["table_index"].empty:
                        table_index_val = table_df["table_index"].iloc[0]

                    if "table_title" in table_df.columns and not table_df["table_title"].empty:
                        table_title_val = table_df["table_title"].iloc[0]

                    # Final clean before saving
                    table_df = drop_metadata_columns(table_df)

                    if table_df.empty:
                        print(f"  Script 2 table {i} skipped (missing required time/value columns after cleaning)")
                        log_rows.append({
                            "source_file": get_excel_basename(selected_source),
                            "sheet_name": sheet_name,
                            "table_index": table_index_val,
                            "table_title": table_title_val,
                            "output_file": "",
                            "method_used": "script_2_fallback",
                            "status": "skipped_missing_time_or_value"
                        })
                        continue

                    table_df.to_csv(table_output_path, index=False)

                    print(f"  Script 2 table {i} saved -> {table_output_name}")
                    print("  Output columns:", list(table_df.columns))

                    log_rows.append({
                        "source_file": get_excel_basename(selected_source),
                        "sheet_name": sheet_name,
                        "table_index": table_index_val,
                        "table_title": table_title_val,
                        "output_file": table_output_name,
                        "method_used": "script_2_fallback",
                        "status": "saved"
                    })
                    saved_any = True

                if not saved_any:
                    print("  No valid tables saved (all missing required time/value columns)")
            else:
                print("  Skipped (both Script 1 and Script 2 returned empty output)")
                log_rows.append({
                    "source_file": get_excel_basename(selected_source),
                    "sheet_name": sheet_name,
                    "table_index": "",
                    "table_title": "",
                    "output_file": "",
                    "method_used": "none",
                    "status": "skipped"
                })

        except Exception as e:
            print(f"  Error processing sheet {sheet_name}: {e}")

            log_rows.append({
                "source_file": get_excel_basename(selected_source),
                "sheet_name": sheet_name,
                "table_index": "",
                "table_title": "",
                "output_file": "",
                "method_used": "error",
                "status": str(e)
            })

    if log_rows:
        log_df = pd.DataFrame(log_rows)
        log_df.to_csv(os.path.join("tidy_outputs", "processing_log.csv"), index=False)
        print("\nSaved log: tidy_outputs/processing_log.csv")

    print("\nFinished processing.")
    print("Final outputs saved in: tidy_outputs/")


if __name__ == "__main__":
    main()

Choose input method:
1. Scrape Excel files from a webpage
2. Use a single local Excel file
3. Use a folder of local Excel files
Invalid choice.

Excel files found:

1. data-annex-ncid-private-motor-insurance-mid-year-2025-settled.xlsx
2. data-annex-ncid-private-motor-insurance-mid-year-2025.xlsx
3. annex-private-motor-insurance-report-7.xlsx
4. data-annex-ncid-private-motor-insurance-mid-year-2024-settled-claims.xlsx
5. data-annex-ncid-private-motor-insurance-mid-year-2024.xlsx
6. annex-ncid-private-motor-insurance-report-6.xlsx
7. data-annex-ncid-private-motor-insurance-mid-year-report-2.xlsx
8. annex-ncid-private-motor-insurance-report-5.xlsx
9. data-annex-ncid-private-motor-insurance-mid-year-report-1.xlsx
10. annex-private-motor-insurance-report-4.xlsx
11. annex-private-motor-insurance-report-3.xlsx
12. annex-private-motor-insurance-report-2-data.xlsx
13. annex-2-private-motor-insurance-report-2.xlsx
14. annex-1---settlement-channels-2015-to-2018.xlsx
15. annex-2---private-motor-in